## 1. Load Data

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as colors

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
df = pd.read_csv('../data/raw/data.csv')

## 2. Data Description

In [ ]:
df.head()

,Period,Year,Month,Quarter,FDI (billion USD),EX (USD),IM (USD),CPI_US (2010 = 100),CPI_VN (2010 = 100),REER (2007M12 = 100),NER,IPI VN (2015 = 100),IPI World (2015 = 100),COVID,WTI Spot Price FOB (Dollars per Barrel),M2 (billion VND)
0,2015-01,2015,1,Q1,1.008955,17273061,16329883,107.177760,143.827689,143.30,21364.54545,100.000000,100.000000,0,47.22,5192194
1,2015-02,2015,2,Q1,1.031874,12096714,12352512,107.643238,143.755775,143.70,21335.20000,90.472674,99.978512,0,50.58,5308012
2,2015-03,2015,3,Q1,1.009171,16912819,17619081,108.283900,143.971409,146.74,21443.45455,94.531437,99.997201,0,47.82,5300817
3,2015-04,2015,4,Q2,1.098022,16684171,15662617,108.504028,144.172969,145.12,21594.68182,93.870811,100.222375,0,54.45,5328232
4,2015-05,2015,5,Q2,1.072688,17462640,17498182,109.057098,144.403646,142.60,21740.19048,95.748314,100.119177,0,59.27,5376435


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 16 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Period                                   132 non-null    object 
 1   Year                                     132 non-null    int64  
 2   Month                                    132 non-null    int64  
 3   Quarter                                  132 non-null    object 
 4   FDI (billion USD)                        132 non-null    float64
 5   EX (USD)                                 132 non-null    int64  
 6   IM (USD)                                 132 non-null    int64  
 7   CPI_US (2010 = 100)                      132 non-null    float64
 8   CPI_VN (2010 = 100)                      132 non-null    float64
 9   REER (2007M12 = 100)                     132 non-null    float64
 10  NER                                      132 non-n

| Variable | Data Type | Description |
| :--- | :--- | :--- |
| `Period` | string | Time period of the observation |
| `Year` | integer | The year of the recorded data |
| `Month` | integer | The month of the recorded data |
| `Quarter` | string | The quarter of the financial/calendar year |
| `FDI (billion USD)` | float | Foreign Direct Investment in billion USD |
| `EX (USD)` | float | Export value in USD |
| `IM (USD)` | float | Import value in USD |
| `CPI_US (2010 = 100)` | float | Consumer Price Index of the United States (Base year 2010) |
| `CPI_VN (2010 = 100)` | float | Consumer Price Index of Vietnam (Base year 2010) |
| `REER (2007M12 = 100)` | float | Real Effective Exchange Rate (Base December 2007) |
| `NER` | float | Nominal Exchange Rate |
| `IPI VN (2015 = 100)` | float | Industrial Production Index of Vietnam (Base year 2015) |
| `IPI World (2015 = 100)` | float | Industrial Production Index of the World (Base year 2015) |
| `COVID` | dummyor | Dummy variable for the COVID-19 pandemic period |
| `WTI Spot Price FOB (Dollars per Barrel)` | float | West Texas Intermediate crude oil spot price per barrel |
| `M2 (billion VND)` | interger | Broad money supply (M2) in billion VND |


## 3. Data Processing

In [ ]:
df = df.drop(columns=["Year", "Month", "Quarter"])

In [ ]:
df["Period"] = pd.to_datetime(df["Period"], format="%Y-%m")
df = df.set_index("Period")
df = df.sort_index()

In [ ]:
df = df.rename(columns={
    "FDI (billion USD)": "FDI",
    "EX (USD)": "EX",
    "IM (USD)": "IM",
    "CPI_US (2010 = 100)": "CPI_US",
    "CPI_VN (2010 = 100)": "CPI_VN",
    "REER (2007M12 = 100)": "REER",
    "IPI VN (2015 = 100)": "IPI_VN",
    "IPI World (2015 = 100)": "IPI_World",
    "WTI Spot Price FOB (Dollars per Barrel)": "WTI",
    "M2 (billion VND)": "M2"
})

In [ ]:
df['TB'] = df['EX']/df['IM']

In [ ]:
log_candidates = ["TB","FDI", "EX", "IM", "WTI", "M2", "NER"]

for col in log_candidates:
    print(col, "min =", df[col].min())

TB min = 0.9241916471545742
FDI min = 1.008955
EX min = 12096714
IM min = 12352512
WTI min = 16.55
M2 min = 5192194
NER min = 21335.2


In [ ]:
df["ln_FDI"] = np.log(df["FDI"])
df["ln_EX"] = np.log(df["EX"])
df["ln_IM"] = np.log(df["IM"])
df["ln_WTI"] = np.log(df["WTI"])
df["ln_M2"] = np.log(df["M2"])
df["ln_TB"] = np.log(df["TB"])

In [ ]:
df['RER'] = df['NER'] * (df['CPI_VN'] / df['CPI_US'])

if (df['RER'] <= 0).any():
    print("Có giá trị RER <= 0, không thể log trực tiếp. Cần kiểm tra dữ liệu.")

df['ln_RER'] = np.log(df['RER'])

print(df[['NER','CPI_VN','CPI_US','RER','ln_RER']].head())

                    NER      CPI_VN      CPI_US           RER     ln_RER
Period                                                                  
2015-01-01  21364.54545  143.827689  107.177760  28670.250346  10.263615
2015-02-01  21335.20000  143.755775  107.643238  28492.809115  10.257407
2015-03-01  21443.45455  143.971409  108.283900  28510.649892  10.258033
2015-04-01  21594.68182  144.172969  108.504028  28693.583673  10.264429
2015-05-01  21740.19048  144.403646  109.057098  28786.414009  10.267659


In [ ]:
df['dln_RER'] = df['ln_RER'].diff()

# Positive shocks: max(Δ,0)
df['ln_RER_pos'] = df['dln_RER'].apply(lambda x: x if x > 0 else 0)

# Negative shocks: min(Δ,0) taken as positive for NARDL
df['ln_RER_neg'] = df['dln_RER'].apply(lambda x: -x if x < 0 else 0)

# Xóa cột trung gian nếu muốn
df.drop(columns=['dln_RER'], inplace=True)

In [ ]:
df['IPI_World_GR'] = df['IPI_World'].pct_change() * 100
df['IPI_World_diff'] = df['IPI_World_GR'].diff()
df['ln_M2_diff'] = np.log(df['ln_M2']).diff()

In [ ]:
unit_root_vars = [
    "ln_FDI",
    "ln_TB",
    "ln_EX",
    "ln_IM",
    "ln_RER_pos",
    "ln_RER_neg",
    "IPI_VN",
    "IPI_World_diff",
    "ln_WTI",
    "ln_M2_diff",
    "COVID"
]

In [ ]:
data_processed = df[unit_root_vars].copy()

In [ ]:
data_processed.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 132 entries, 2015-01-01 to 2025-12-01
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ln_FDI          132 non-null    float64
 1   ln_TB           132 non-null    float64
 2   ln_EX           132 non-null    float64
 3   ln_IM           132 non-null    float64
 4   ln_RER_pos      132 non-null    float64
 5   ln_RER_neg      132 non-null    float64
 6   IPI_VN          132 non-null    float64
 7   IPI_World_diff  130 non-null    float64
 8   ln_WTI          132 non-null    float64
 9   ln_M2_diff      131 non-null    float64
 10  COVID           132 non-null    int64  
dtypes: float64(10), int64(1)
memory usage: 12.4 KB


In [ ]:
data_processed.to_csv('..\data/processed/data_processed.csv', index=False, encoding='utf-8-sig')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\admin\AppData\Local\Temp\ipykernel_33604\3722799784.py:1: SyntaxWarning: invalid escape sequence '\d'
  data_processed.to_csv('..\data/processed/data_processed.csv', index=False, encoding='utf-8-sig')
